## Identifying Roadway Roughness Near Public At-Grade Railroad Crossings in Oklahoma

Jeff Smith

This project uses Oklahoma Department of Transportation pavement roughness data and Federal Railroad Administration railroad crossing data to identify roadway segments with elevated International Roughness Index (IRI) values near public at-grade railroad crossings in Oklahoma.

## Part 01 - Project Setup

Import the required Python modules and define the paths to the raw input datasets and project output.

In [ ]:
from pathlib import Path
import arcpy

# Project folders
projectFolder = Path.cwd()
rawDataFolder = projectFolder / "RawData"
outputFolder = projectFolder / "Output"

# Input datasets
pavementIRI = rawDataFolder / "OK_Pavement_IRI.shp"
railroadCrossings = rawDataFolder / "RailRoadCrossings.shp"

# Output GeoPackage
outputGpkg = outputFolder / "RailRoughnessAnalysis.gpkg"

# Confirm paths
print("Project folder:", projectFolder)
print("Raw data folder:", rawDataFolder)
print("Output folder:", outputFolder)

print("\nPavement IRI exists:", pavementIRI.exists())
print("Railroad crossings exists:", railroadCrossings.exists())

## Part 02 - Inspect Input Data

Review the basic properties and attribute fields of the pavement IRI and railroad crossing datasets before beginning the analysis.

In [13]:
# Inspect pavement IRI data
pavementDesc = arcpy.Describe(str(pavementIRI))

print("PAVEMENT IRI")
print("Feature type:", pavementDesc.shapeType)
print("Spatial reference:", pavementDesc.spatialReference.name)
print("Record count:", arcpy.management.GetCount(str(pavementIRI))[0])

print("\nFields:")
for field in arcpy.ListFields(str(pavementIRI)):
    print(field.name, "-", field.type)


# Inspect railroad crossing data
crossingDesc = arcpy.Describe(str(railroadCrossings))

print("\nRAILROAD CROSSINGS")
print("Feature type:", crossingDesc.shapeType)
print("Spatial reference:", crossingDesc.spatialReference.name)
print("Record count:", arcpy.management.GetCount(str(railroadCrossings))[0])

print("\nFields:")
for field in arcpy.ListFields(str(railroadCrossings)):
    print(field.name, "-", field.type)

PAVEMENT IRI
Feature type: Polyline
Spatial reference: GCS_WGS_1984
Record count: 156932

Fields:
FID - OID
Shape - Geometry
FROMMEASUR - Double
TOMEASURE - Double
IRI_AVG - Integer
PERCENTCRA - Integer
FAULT_AVG - Double
RUT_AVG - Double
LOC_ERROR - String
ROUTE_ID - String
TENTH_SEGM - String
COLL_DT - String
FROMMEAS_1 - Double
TOMEASUR - Double
DIRECTION - String
DIVISION - String
GRP - String
LANES - Double
GRADE - Double
CURVE - Double
LRS_DT - String
IRI_CODE - String
RUTTING_CO - String
FAULTING_C - String
CRACKING_P - String
NFC - Double
URBAN_AREA - Double
PAVEMENT_T - String
Shape__Len - Double

RAILROAD CROSSINGS
Feature type: Point
Spatial reference: GCS_WGS_1984
Record count: 242108

Fields:
FID - OID
Shape - Geometry
CrossingID - String
ReasonCode - String
StateCode - String
STATENAME - String
CountyCode - String
COUNTYNAME - String
CityCode - String
CITYNAME - String
StateAbbre - String
RailroadCo - String
RailroadDi - String
RailroadSu - String
HighwayNam - String
STRE

## Part 03 - Review Filtering Values

Review the unique values in the railroad crossing fields that will be used to identify public at-grade crossings in Oklahoma.

In [14]:
# Review values used to filter railroad crossings
filterFields = ["StateAbbre", "CrossingTy", "CrossingPo", "CrossingPu"]

for fieldName in filterFields:
    values = set()

    with arcpy.da.SearchCursor(str(railroadCrossings), [fieldName]) as cursor:
        for row in cursor:
            values.add(row[0])

    print(f"{fieldName}:")
    print(sorted(values, key=lambda x: str(x)))
    print()

StateAbbre:
['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']

CrossingTy:
[' ', 'Private', 'Public']

CrossingPo:
['At Grade', 'RR Over', 'RR Under']

CrossingPu:
['Highway', 'Pathway,Ped.', 'Station,Ped.']



## Part 04 - Create Project Output

Create the output folder and GeoPackage that will store the processed project layers.

In [15]:
# Create output folder if it does not already exist
outputFolder.mkdir(exist_ok=True)

# Create GeoPackage
if not outputGpkg.exists():
    arcpy.management.CreateSQLiteDatabase(
        str(outputGpkg),
        "GEOPACKAGE"
    )

    print("GeoPackage created.")
else:
    print("GeoPackage already exists.")

GeoPackage already exists.


## Part 05 - Filter Railroad Crossings

Filter the national railroad crossing dataset to include only public highway at-grade railroad crossings located in Oklahoma. The filtered features will be stored in the project GeoPackage for use in the remaining analysis.

In [16]:
import railroadRoughness as project_module

# Define the filtered railroad crossing output
filteredCrossings = outputGpkg / "OK_Public_AtGrade_Crossings"

# Filter the national crossing dataset
project_module.filter_crossings(
    railroadCrossings,
    filteredCrossings
)

# Confirm the output and display the number of selected crossings
if arcpy.Exists(str(filteredCrossings)):
    filteredCount = arcpy.management.GetCount(str(filteredCrossings))[0]

    print("Filtered crossing layer created successfully.")
    print("Oklahoma public highway at-grade crossings:", filteredCount)
else:
    print("Filtered crossing layer was not created.")

Oklahoma public at-grade railroad crossings created.
Filtered crossing layer created successfully.
Oklahoma public highway at-grade crossings: 3632


## Part 06 - User Input

Ask the user to enter the analysis distance around each railroad crossing. This distance will be used to create crossing buffers and identify nearby pavement IRI segments. All IRI values within the analysis area will be retained for later comparison.

In [17]:
# Ask the user for the railroad crossing buffer distance
while True:
    bufferDistance = float(input("Enter the railroad crossing buffer distance in feet: "))

    if bufferDistance > 0:
        break
    else:
        print("Buffer distance must be greater than 0.")

print("\nAnalysis distance:", bufferDistance, "feet")


Analysis distance: 10.0 feet


## Part 07 - Buffer Railroad Crossings

Create a buffer around each Oklahoma public highway at-grade railroad crossing using the user-defined analysis distance. A geodesic buffer is used because the input data is stored in the WGS 1984 geographic coordinate system.

In [18]:
# Define the crossing buffer output
crossingBuffers = outputGpkg / "Railroad_Crossing_Buffers"

# Create the crossing buffers
project_module.buffer_crossings(
    filteredCrossings,
    crossingBuffers,
    bufferDistance
)

# Confirm the output
if arcpy.Exists(str(crossingBuffers)):
    bufferCount = arcpy.management.GetCount(str(crossingBuffers))[0]

    print("Crossing buffer layer created successfully.")
    print("Buffer polygons:", bufferCount)
else:
    print("Crossing buffer layer was not created.")

Railroad crossings buffered by 10.0 feet.
Crossing buffer layer created successfully.
Buffer polygons: 3632


## Part 08 - Identify Pavement Segments Near Railroad Crossings

Use a spatial join to identify pavement IRI segments that intersect the railroad crossing buffers. The complete pavement segments and their attributes are retained so their IRI values and roadway positions can be compared with neighboring segments later in the analysis.

In [19]:
# Define the pavement segment output
crossingSegments = outputGpkg / "IRI_Segments_Near_Railroad_Crossings"

# Identify pavement segments that intersect crossing buffers
project_module.identify_crossing_segments(
    pavementIRI,
    crossingBuffers,
    crossingSegments
)

# Confirm the output
if arcpy.Exists(str(crossingSegments)):
    crossingSegmentCount = arcpy.management.GetCount(str(crossingSegments))[0]

    print("Crossing-related pavement segment layer created successfully.")
    print("Spatial join records:", crossingSegmentCount)
else:
    print("Crossing-related pavement segment layer was not created.")

Pavement segments near railroad crossings identified.
Crossing-related pavement segment layer created successfully.
Spatial join records: 176


## Part 09 - Identify Before and After Pavement Segments

Group the pavement IRI segments by route and direction and sort each group by roadway measure. For each pavement segment associated with a railroad crossing, identify the segment immediately before and immediately after it. This keeps the comparison on the same route and in the same roadway direction.

In [20]:
# Define the output for crossing and neighboring pavement segments
railAnalysisSegments = outputGpkg / "Railroad_IRI_Analysis"

# Identify crossing, before, and after pavement segments
project_module.flag_neighbor_segments(
    pavementIRI,
    crossingSegments,
    railAnalysisSegments
)

# Confirm the output
if arcpy.Exists(str(railAnalysisSegments)):
    analysisCount = arcpy.management.GetCount(str(railAnalysisSegments))[0]

    print("Railroad IRI analysis layer created successfully.")
    print("Analysis pavement segments:", analysisCount)
else:
    print("Railroad IRI analysis layer was not created.")

Crossing, before, and after pavement segments identified.
Railroad IRI analysis layer created successfully.
Analysis pavement segments: 479


## Part 10 - Remove Invalid IRI Values

Remove pavement segments with an IRI value of zero before calculating summary statistics. These records are retained while identifying the immediate before and after segments so that the roadway sequence remains intact, but they are excluded from the final roughness comparison because a zero value does not represent a useful IRI measurement for this analysis.

In [21]:
# Define the valid IRI analysis output
validIRISegments = outputGpkg / "Railroad_IRI_Analysis_Valid"

# Remove records with an IRI value of zero
project_module.filter_valid_iri(
    railAnalysisSegments,
    validIRISegments
)

# Compare record counts
originalCount = int(
    arcpy.management.GetCount(str(railAnalysisSegments))[0]
)

validCount = int(
    arcpy.management.GetCount(str(validIRISegments))[0]
)

removedCount = originalCount - validCount

print("Original analysis segments:", originalCount)
print("Valid IRI segments:", validCount)
print("Zero IRI segments removed:", removedCount)

Pavement segments with valid IRI values selected.
Original analysis segments: 479
Valid IRI segments: 458
Zero IRI segments removed: 21


## Part 11 - Final IRI Summary and Review

Calculate and display summary statistics for valid IRI values on railroad crossing segments and the pavement segments immediately before and after them. Compare the average IRI values to determine whether roadway roughness tends to increase at railroad crossing locations.

In [22]:
# Define the summary statistics output
iriSummary = outputGpkg / "IRI_Summary_By_RailFlag"

# Calculate IRI summary statistics using only valid IRI values
project_module.summarize_iri(
    validIRISegments,
    iriSummary
)

# Confirm the output
if arcpy.Exists(str(iriSummary)):
    print("IRI summary statistics calculated successfully.")

    flagNames = {
        1: "Railroad Crossing",
        2: "Before Crossing",
        3: "After Crossing"
    }

    summaryFields = [
        "RailFlag",
        "COUNT_IRI_AVG",
        "MEAN_IRI_AVG",
        "MIN_IRI_AVG",
        "MAX_IRI_AVG",
        "STD_IRI_AVG"
    ]

    meanValues = {}

    print("\nIRI Summary")
    print()

    with arcpy.da.SearchCursor(str(iriSummary), summaryFields) as cursor:
        for railFlag, count, meanIRI, minIRI, maxIRI, stdIRI in cursor:

            flagName = flagNames.get(railFlag, "Unknown")

            print(f"\n{flagName}")
            print("Count:", count)
            print(f"Mean IRI: {meanIRI:.2f}")
            print("Minimum IRI:", minIRI)
            print("Maximum IRI:", maxIRI)
            print(f"Standard Deviation: {stdIRI:.2f}")

            meanValues[railFlag] = meanIRI

    # Compare the crossing mean with the neighboring pavement segments
    if 1 in meanValues and 2 in meanValues and 3 in meanValues:

        crossingMean = meanValues[1]
        beforeMean = meanValues[2]
        afterMean = meanValues[3]

        neighboringMean = (beforeMean + afterMean) / 2
        iriDifference = crossingMean - neighboringMean

        print("\nBasic Analysis")
        print()
        print(f"Crossing mean IRI: {crossingMean:.2f}")
        print(f"Average neighboring IRI: {neighboringMean:.2f}")
        print(f"Difference: {iriDifference:.2f}")

        if crossingMean > neighboringMean:
            print("The average IRI is higher at railroad crossing segments than on the neighboring pavement segments.")
        elif crossingMean < neighboringMean:
            print("The average IRI is lower at railroad crossing segments than on the neighboring pavement segments.")
        else:
            print("The average IRI is the same at railroad crossing segments and neighboring pavement segments.")

else:
    print("IRI summary table was not created.")

IRI summary statistics calculated.
IRI summary statistics calculated successfully.

IRI Summary


Railroad Crossing
Count: 165
Mean IRI: 235.49
Minimum IRI: 64
Maximum IRI: 488
Standard Deviation: 80.05

Before Crossing
Count: 148
Mean IRI: 164.66
Minimum IRI: 48
Maximum IRI: 509
Standard Deviation: 86.34

After Crossing
Count: 145
Mean IRI: 146.94
Minimum IRI: 44
Maximum IRI: 444
Standard Deviation: 75.83

Basic Analysis

Crossing mean IRI: 235.49
Average neighboring IRI: 155.80
Difference: 79.69
The average IRI is higher at railroad crossing segments than on the neighboring pavement segments.
